# Lab: Logistic Regression for Customer Churn Predictions

---

## 1. Introduction

In this lab, we'll step into the shoes of a data scientist at a telecommunications company. The company is concerned about customers leaving for competitiors and wants to identify which customers are at a high risk of "churning" (leaving).

* **Goal:** To build a logistic regression model that predicts the probability of a customer churning based on their demographic and service usage data.
* **Data:** A historical dataset of telecom customers, where each row represents a customer and includes details about their services and whether they churned.

---

## 2. Setup and Data Loading

We'll start by importing the necessary libraries. We'll use `pandas` for data handling, `numpy` for numerical operations, `matplotlib` and `seaborn` for plotting, and `sklearn` for our modeling and evaluation tools.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, jaccard_score, f1_score, log_loss, confusion_matrix

In [2]:
# Load the data from the URL
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/ChurnData.csv"
churn_df = pd.read_csv(url)

# Display a sample of the data
churn_df.sample(5)

,tenure,age,address,income,ed,employ,equip,callcard,wireless,longmon,tollmon,equipmon,cardmon,wiremon,longten,tollten,cardten,voice,pager,internet,callwait,confer,ebill,loglong,logtoll,lninc,custcat,churn
124,59.0,55.0,29.0,42.0,3.0,21.0,0.0,1.0,0.0,8.95,0.0,0.00,15.25,0.00,492.35,0.00,855.0,0.0,0.0,0.0,0.0,0.0,0.0,2.192,3.240,3.738,2.0,0.0
164,72.0,61.0,34.0,61.0,1.0,8.0,0.0,1.0,0.0,27.35,0.0,0.00,13.25,0.00,1980.15,0.00,895.0,0.0,0.0,0.0,0.0,0.0,0.0,3.309,3.240,4.111,2.0,0.0
149,19.0,35.0,7.0,58.0,3.0,5.0,1.0,1.0,1.0,3.65,37.0,40.30,21.25,43.05,65.15,690.85,385.0,1.0,1.0,1.0,1.0,1.0,0.0,1.295,3.611,4.060,4.0,1.0
1,33.0,33.0,12.0,33.0,2.0,0.0,0.0,0.0,0.0,9.45,0.0,0.00,0.00,0.00,288.80,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.246,3.240,3.497,1.0,1.0
161,64.0,37.0,10.0,44.0,4.0,9.0,1.0,1.0,0.0,16.15,0.0,35.05,22.00,0.00,965.30,0.00,1350.0,1.0,0.0,0.0,0.0,0.0,1.0,2.782,3.240,3.784,2.0,0.0



---

## 3. Data Preprocessing and Exploration

Our first step is to prepare the data for the model. This involves selecting relevant features and ensuring they are in the correct format.

### 3.1 Feature Selection
For this model, we'll select a subset of numerical features that might influence a customer's decision to churn, such as their tenure with the company, age, income, and service usage.

In [3]:
# Select the features we'll use for the model
churn_df = churn_df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip', 'churn']]

# The 'churn' column is our target. Scikit-learn algorithms require the target to be numerical
# Let's convert it to an integer type (0 or 1)
churn_df['churn'] = churn_df['churn'].astype('int')

churn_df.head()

,tenure,age,address,income,ed,employ,equip,churn
0,11.0,33.0,7.0,136.0,5.0,5.0,0.0,1
1,33.0,33.0,12.0,33.0,2.0,0.0,0.0,1
2,23.0,30.0,9.0,30.0,1.0,2.0,0.0,0
3,38.0,35.0,5.0,76.0,2.0,10.0,1.0,0
4,7.0,35.0,14.0,80.0,2.0,15.0,0.0,0


### 3.2 Define Feature Matrix (X) and Target Vector (y)
Now we separate our data into the features (`X`) that will be used for prediction and the target (`y`) that we want to predict.

In [4]:
# X contains our independent variables (features)
X = churn_df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip']].values

# y contains our dependent variable (the target)
y = churn_df['churn'].values

### 3.3 Create Train and Test Datasets

We must split our data before any further processing (like scaling). This ensures that our test set remains a truly "unseen" dataset for evaluating our model's performance.

In [5]:
# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 3.4. Standardize the Features
Our features are on different scales (e.g. `income` can be in the tens of thousands, while `age` is in double digits). We'll standardize them to have a mean of 0 and a standard deviation of 1.

**Best Practice:** We `fit` the scaler **only on the training data** to learn its parameters (mean and standard deviation). Then we use that *same* fitted scaler to `transform` both the training and test data. This prevents data leakage from the test set into our training process.

In [6]:
scaler = StandardScaler()

# Fit the scaler on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler to transform the test data
X_test_scaled = scaler.transform(X_test)